## Manipulación de Datos Precio Interno del Café

    Convertimos el precio interno diario del bulto de café al promedio de precio semanal.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:

# ---------------------------------------------------------
# 1. Leer el archivo de la FNC
# ---------------------------------------------------------
fnc_path = Path(r"C:\Users\junio\OneDrive\Escritorio\Andes\MIID\Gerencia Proyectos\Anteproyecto\data\FNC")
file_precios = fnc_path / "Precio_act.xlsx"

df = pd.read_excel(file_precios)


In [3]:

# Limpiamos nombres de columnas
df.columns = df.columns.str.strip()

# Ajusta estos nombres exactos si en el archivo cambian un poco
df = df.rename(columns={
    "Fecha": "date",
    "Precio Interno ($/125 Kg)": "precio_interno"
})

In [4]:

# ---------------------------------------------------------
# 2. Asegurar tipos: fecha y precio numérico
# ---------------------------------------------------------
# Fecha a datetime (Excel suele traerlo bien, pero por si acaso)
df["date"] = pd.to_datetime(df["date"])

In [5]:

# Precio a número:
# - si viene como texto tipo "1.073.000", quitamos puntos de miles
# - si hay comas como decimales, las pasamos a punto
df["precio_interno"] = (
    df["precio_interno"]
    .astype(str)
    .str.replace(".", "", regex=False)   # quita miles
    .str.replace(",", ".", regex=False)  # por si acaso
    .astype(float)
)


In [6]:
# ---------------------------------------------------------
# 3. Promedio semanal del precio
# ---------------------------------------------------------
df = df.set_index("date").sort_index()

precio_semanal = (
    df
    .resample("W")            # misma lógica que usamos para el clima
    .mean(numeric_only=True)  # promedio semanal
    .reset_index()
)


In [7]:
# (Opcional) añadir año y semana ISO para análisis / merge
iso = precio_semanal["date"].dt.isocalendar()
precio_semanal["year_iso"] = iso.year
precio_semanal["week_iso"] = iso.week

In [8]:
# ---------------------------------------------------------
# 4. Guardar resultado
# ---------------------------------------------------------
precio_semanal.to_excel(fnc_path / "precio_interno_semanal.xlsx", index=False)
# o en CSV si prefieres:
# precio_semanal.to_csv(fnc_path / "precio_interno_semanal.csv", index=False)

print("Precio_interno_semanal generado en:", fnc_path)

Precio_interno_semanal generado en: C:\Users\junio\OneDrive\Escritorio\Andes\MIID\Gerencia Proyectos\Anteproyecto\data\FNC
